# Breakout Room #3 — Recomendar: filtrado colaborativo por ítem y por cliente

**Objetivo:**
- Recomendar productos con la matriz del breakout anterior, de dos maneras: **por ítem** y **por cliente**.
- Entender *por qué* cada método recomienda lo que recomienda.
- Comparar costo, estabilidad y qué pasa con clientes sin historia.

Dos ideas, la misma matriz:

| | Pregunta que contesta | Cómo puntúa un producto candidato |
|---|---|---|
| **Por ítem** (item-based) | "A quien compró X, ¿qué otros productos *se parecen* a lo que ya compró?" | suma de la similitud entre el candidato y **cada producto que el cliente ya tiene** |
| **Por cliente** (user-based) | "¿Qué compraron los clientes *parecidos a mí* que yo no he comprado?" | suma de lo que compraron los **K vecinos más parecidos**, ponderada por su similitud |

Donde dice **`# TODO`** hay algo para que ustedes completen.

In [1]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
VERDE, OSCURO, NARANJA, GRIS = "#00704A", "#11241C", "#C4512F", "#9A988E"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})

orders = pd.read_csv("data/orders.csv", parse_dates=["order_date"])
customers = pd.read_csv("data/customers.csv")
CUTOFF = orders["order_date"].max() - pd.Timedelta(days=90)
train = orders[orders["order_date"] <= CUTOFF]

# La matriz del breakout anterior (1 = el cliente compró ese producto antes del corte)
M = (pd.crosstab(train["customer_id"], train["item"]) > 0).astype(int).reindex(customers["customer_id"], fill_value=0)
X = M.values.astype(float)
productos = list(M.columns)
print("Matriz:", M.shape)

Matriz: (2000, 29)


## Paso 1 — las similitudes

Dos matrices de similitud coseno: **productos × productos** (29 × 29) y **clientes × clientes** (2,000 × 2,000).
Ponemos la diagonal en 0 (un producto o cliente no es "vecino" de sí mismo). Fíjense en cuánto tarda cada una.

In [2]:
t0 = time.perf_counter(); S_item = cosine_similarity(X.T); np.fill_diagonal(S_item, 0); t_item = time.perf_counter() - t0
t0 = time.perf_counter(); S_user = cosine_similarity(X);   np.fill_diagonal(S_user, 0); t_user = time.perf_counter() - t0
print(f"Matriz producto×producto: {S_item.shape}  → {S_item.size:,} celdas · {t_item*1000:.1f} ms")
print(f"Matriz cliente×cliente:   {S_user.shape}  → {S_user.size:,} celdas · {t_user*1000:.1f} ms")

Matriz producto×producto: (29, 29)  → 841 celdas · 20.6 ms
Matriz cliente×cliente:   (2000, 2000)  → 4,000,000 celdas · 22.9 ms


## Paso 2 — recomendar por ítem

Para un cliente, el puntaje de cada producto es la suma de su similitud con **los productos que el cliente ya compró**.
Ya excluimos lo que el cliente tiene (no se recomienda lo que ya compró) y el **Agua** (ya vimos que no distingue a nadie).

In [3]:
def top_k(scores, cid, k=3):
    owned = M.loc[cid].values > 0
    s = pd.Series(scores, index=productos).where(~owned).drop("Agua", errors="ignore")
    return s.sort_values(ascending=False).head(k)

def puntaje_item(cid):
    u = M.loc[cid].values.astype(float)         # lo que el cliente compró (vector de 0/1)
    # TODO 1: puntaje de cada producto = suma de su similitud con lo que el cliente ya compró.
    # Pista: es un producto punto (matricial) entre u y S_item.
    return u @ S_item

for cid in (14, 62):
    print(f"Cliente {cid} compró: {', '.join(M.columns[M.loc[cid] == 1])}")
    print(top_k(puntaje_item(cid), cid).round(2).to_string(), "\n")

Cliente 14 compró: Agua, Banana Bread, Cappuccino, Flat White, Muffin, Té Chai
Té Verde     2.03
Croissant    1.69
Latte        1.56 

Cliente 62 compró: Brownie, Cold Brew, Ensalada, Granola, Matcha Latte
Copa de Fruta         1.57
Yogur                 1.33
Chocolate Caliente    1.02 



## Paso 3 — recomendar por cliente

Buscamos a los **K = 30 clientes más parecidos** y sumamos lo que compraron, ponderado por cuánto se parecen.

In [4]:
K_VEC = 30

def puntaje_user(cid):
    i = M.index.get_loc(cid)
    sims = S_user[i]                              # similitud de este cliente con todos los demás
    vecinos = np.argsort(-sims)[:K_VEC]           # los K más parecidos
    # TODO 2: puntaje de cada producto = suma de lo que compraron los vecinos, ponderada por su similitud.
    # Pista: sims[vecinos] (un vector de K pesos) por X[vecinos] (K filas de la matriz).
    return sims[vecinos] @ X[vecinos]

for cid in (14, 62):
    print(f"Cliente {cid}:")
    print(top_k(puntaje_user(cid), cid).round(2).to_string(), "\n")

Cliente 14:
Té Verde     7.29
Croissant    2.17
Galleta      1.45 

Cliente 62:
Copa de Fruta    5.63
Yogur            5.57
Refresher        2.70 



## Paso 4 — ¿por qué recomienda eso? (explicabilidad)

Un recomendador que no se puede explicar es difícil de defender ante el negocio. Para el cliente 14, miremos el **motivo** de la primera recomendación de cada método.

In [5]:
cid = 14
rec_i = top_k(puntaje_item(cid), cid, 1).index[0]
rec_u = top_k(puntaje_user(cid), cid, 1).index[0]

# Por ítem: ¿de qué productos que ya compró viene el puntaje?
j = productos.index(rec_i)
tiene = [p for p in productos if M.loc[cid, p] == 1 and p != "Agua"]
motivo_i = pd.Series({p: S_item[productos.index(p), j] for p in tiene}).sort_values(ascending=False)
print(f"POR ÍTEM → '{rec_i}' porque se parece a lo que ya compró:")
print(motivo_i.round(2).head(4).to_string(), "\n")

# Por cliente: ¿qué vecinos lo compraron?
i = M.index.get_loc(cid)
vec = np.argsort(-S_user[i])[:K_VEC]
con_prod = [(int(M.index[v]), round(float(S_user[i, v]), 2)) for v in vec if M.iloc[v][rec_u] == 1][:4]
print(f"POR CLIENTE → '{rec_u}' porque lo compraron vecinos como (cliente, similitud): {con_prod}")
print(f"   {sum(M.iloc[v][rec_u] for v in vec)} de {K_VEC} vecinos lo compraron.")

POR ÍTEM → 'Té Verde' porque se parece a lo que ya compró:
Té Chai         0.58
Banana Bread    0.52
Muffin          0.31
Cappuccino      0.10 

POR CLIENTE → 'Té Verde' porque lo compraron vecinos como (cliente, similitud): [(941, 0.73), (1778, 0.73), (1800, 0.73), (1852, 0.73)]
   10 de 30 vecinos lo compraron.


## Paso 5 — ¿coinciden los dos métodos? ¿cuánto cuesta cada uno?

In [6]:
con_hist = M.index[M.sum(axis=1) >= 2]
col_agua = productos.index("Agua")
def mejor(sc, owned):
    s = np.where(owned > 0, -np.inf, sc); s[col_agua] = -np.inf
    return np.argsort(-s)[:3]

coinc1, coinc3 = [], []
t0 = time.perf_counter()
for cid in con_hist:
    owned = M.loc[cid].values
    ti, tu = mejor(puntaje_item(cid), owned), mejor(puntaje_user(cid), owned)
    coinc1.append(ti[0] == tu[0]); coinc3.append(len(set(ti) & set(tu)))
dt = time.perf_counter() - t0
print(f"Clientes con al menos 2 productos: {len(con_hist):,}")
print(f"Misma recomendación #1: {np.mean(coinc1):.0%}")
print(f"Coinciden en al menos 2 de las 3 primeras: {np.mean(np.array(coinc3) >= 2):.0%}")
print(f"Tiempo de recomendar a todos (los dos métodos): {dt:.1f} s")

Clientes con al menos 2 productos: 1,669
Misma recomendación #1: 68%
Coinciden en al menos 2 de las 3 primeras: 82%
Tiempo de recomendar a todos (los dos métodos): 0.3 s


## Paso 6 — los casos difíciles

¿Qué le recomienda cada método a un cliente con **un solo producto** y a uno con **cero**?

In [7]:
uno = M.index[M.sum(axis=1) == 1][0]
cero = M.index[M.sum(axis=1) == 0][0]
pop = pd.Series(X.sum(axis=0), index=productos)

print(f"Cliente {uno} (compró: {', '.join(M.columns[M.loc[uno] == 1])})")
print("  por ítem   :", list(top_k(puntaje_item(uno), uno).index))
print("  por cliente:", list(top_k(puntaje_user(uno), uno).index))
print(f"\nCliente {cero} (0 productos)")
print("  por ítem   :", list(top_k(puntaje_item(cero), cero).round(2).items()))
print("  por cliente:", list(top_k(puntaje_user(cero), cero).round(2).items()))
print("  (todo en 0: el método no tiene nada que ordenar)")
print("\nRespaldo común: lo más popular →", list(pop.drop("Agua").sort_values(ascending=False).head(3).index))

Cliente 30 (compró: Refresher)
  por ítem   : ['Frappuccino', 'Pastel de Queso', 'Galleta']
  por cliente: ['Pastel de Queso', 'Galleta', 'Frappuccino']

Cliente 11 (0 productos)
  por ítem   : [('Americano', 0.0), ('Bagel', 0.0), ('Banana Bread', 0.0)]
  por cliente: [('Americano', 0.0), ('Bagel', 0.0), ('Banana Bread', 0.0)]
  (todo en 0: el método no tiene nada que ordenar)

Respaldo común: lo más popular → ['Chocolate Caliente', 'Galleta', 'Té Chai']


1. Los dos métodos, ¿recomiendan lo mismo? ¿En qué casos preferirían cada uno?
2. Tiempos y tamaños del Paso 1: si la base fuera de 2 millones de clientes y 10,000 productos, ¿cuál de las dos matrices de similitud podrían calcular y guardar de antemano?
3. ¿Qué hacen con el cliente de 0 productos? ¿Y con un producto *nuevo* que nadie ha comprado todavía?
4. ¿cómo sabrían cuál de los dos recomienda *mejor*?

## Respuestas — para la clase

**Resultados:** cliente 14 → por ítem **Té Verde, Croissant, Latte**; por cliente **Té Verde, Croissant, Galleta**. Cliente 62 → ambos recomiendan **Copa de Fruta y Yogur** (perfil saludable). Coinciden en la #1 en el **68%** de los 1,669 clientes con ≥ 2 productos y en al menos 2 de 3 en el **82%**.

**1. ¿Recomiendan lo mismo? ¿Cuándo preferir cada uno?**
Mayormente sí (68% / 82%), porque los dos salen de la misma matriz y los segmentos están muy marcados. Difieren en la 3.ª posición y en clientes "mezclados".
- **Por ítem:** cuando importa **explicar** ("porque compraste Té Chai y Banana Bread"), cuando el catálogo es más chico que la base de clientes, y cuando hay que **precalcular** (la matriz de productos cambia poco). Es lo que usaría para el carrusel y la caja.
- **Por cliente:** captura gustos combinados que no están en un par de productos (puede traer **Galleta**, que no se parece a nada de lo del cliente 14 pero sí la compran sus vecinos) → más descubrimiento. Útil con base chica y catálogo grande, pero es más inestable (los vecinos cambian con cada compra) y más difícil de defender ("lo compraron los clientes 941 y 1778").

**2. Con 2 millones de clientes y 10,000 productos, ¿cuál matriz se puede precalcular?**
La de **productos (10,000 × 10,000 = 10⁸ celdas, ~0.8 GB)**. Además es estable: se recalcula una vez por noche/semana. La de clientes sería 2M × 2M = **4×10¹² celdas (~32 TB)**, crece al cuadrado con la base y cambia cada día. Ojo: los 22 ms de hoy engañan, porque el costo de cliente-based crece **cuadráticamente** con los clientes; con 2,000 clientes todavía cabe en memoria, con 2M no.

**3. ¿Qué hacer con el cliente de 0 productos? ¿Y con un producto nuevo?**
- **Cliente sin historia:** ambos métodos dan puntaje 0 a todo (el orden que sale es alfabético, sin sentido). Respaldo: **popularidad** (Chocolate Caliente, Galleta, Té Chai), o mejor, popularidad dentro de su segmento (edad, canal, tienda) si se conoce. Apenas compre 1–2 productos se pasa a por ítem.
- **Producto nuevo:** columna en ceros, similitud 0 con todo, **nunca se recomienda** (arranque en frío del ítem). Hay que empujarlo por fuera del colaborativo: contenido/atributos (categoría, precio, "bebida fría dulce" → parecido a Frappuccino), un espacio fijo de "Nuevo" en el carrusel o exploración aleatoria controlada hasta juntar compras.

**4. ¿Cómo sabrían cuál recomienda mejor?**
Dos niveles:
1. **Offline (holdout):** esconder los últimos 90 días, recomendar con el pasado y medir hit rate@3, precisión@3 y cobertura (Breakout 4).
2. **Online (A/B con control):** el holdout solo dice quién **acierta** lo que el cliente iba a comprar igual. Para saber cuál **genera** ventas extra hay que mostrarlo a un grupo aleatorio y comparar ARPU contra un control que no lo ve. Esta es la que decide.
